# Solution C — APIM → Event Hub → Blob (Capture) 測試

**目的**：驗證 Solution C 完整 body 落地能力，並與 Solution ① 雙軌獨立。

**隔離原則**：
- 只有送出 header `X-Logging-Channel: solution-c` 的請求才會觸發 EH 寫入
- Solution ①（AppInsights）對所有流量持續啟用，互不干擾
- 每筆請求帶 `X-Run-Id`（=correlationId），用來事後在 EH/Blob/ADX 拼回完整對話

**前置**：
1. `.\scripts\deploy-eventhub-logging.ps1` ✅
2. `.\scripts\apply-eventhub-policy.ps1 -ApiName kunlenewfoundry01` ✅

In [ ]:
import os, time, json, subprocess
from openai import AzureOpenAI

APIM_ENDPOINT     = 'https://testaigw01.azure-api.net'
APIM_API_PATH     = '/kunlenewfoundry01/openai/v1/'
APIM_SUBSCRIPTION = os.environ.get('APIM_SUBSCRIPTION_KEY', '<your-apim-key>')
DEPLOYMENT_NAME   = 'Kimi-K2.5'

RUN_ID = f'solc-{int(time.time())}'
print('RUN_ID =', RUN_ID)

client = AzureOpenAI(
    azure_endpoint = APIM_ENDPOINT + APIM_API_PATH,
    api_key        = APIM_SUBSCRIPTION,
    api_version    = '2024-10-21',
    default_headers= {
        'X-Logging-Channel': 'solution-c',
        'X-Run-Id':          RUN_ID
    },
)

## TC-C1 — Non-streaming 短回覆（驗 happy path）

In [ ]:
marker = f'{RUN_ID}-c1'
resp = client.chat.completions.create(
    model = DEPLOYMENT_NAME,
    messages = [{'role':'user','content': f'[{marker}] 用一句話說明 Event Hub'}],
    max_tokens = 200,
    extra_headers = {'X-Run-Id': marker},
)
print('Marker :', marker)
print('Status :', 'OK')
print('Usage  :', resp.usage)
print('Content:', resp.choices[0].message.content[:200])

## TC-C2 — 大型回覆（驗證超過 256KB 不被截斷，多 chunk）

In [ ]:
marker = f'{RUN_ID}-c2'
resp = client.chat.completions.create(
    model = DEPLOYMENT_NAME,
    messages = [{'role':'user','content': f'[{marker}] 用 reasoning 詳細解釋 Kubernetes 從零到生產所有概念，至少 8000 字繁中'}],
    max_tokens = 8000,
    extra_headers = {'X-Run-Id': marker},
)
print('Marker     :', marker)
print('Resp bytes :', len((resp.choices[0].message.content or '').encode('utf-8')))
print('Usage      :', resp.usage)

## TC-C3 — Streaming SSE（驗證 R4 場景下方案 C 是否拿得到完整 body）

In [ ]:
marker = f'{RUN_ID}-c3'
stream = client.chat.completions.create(
    model = DEPLOYMENT_NAME,
    messages = [{'role':'user','content': f'[{marker}] streaming 解釋 reasoning model 如何推理'}],
    max_tokens = 4000,
    stream = True,
    stream_options = {'include_usage': True},
    extra_headers = {'X-Run-Id': marker},
)
chars = 0
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        chars += len(chunk.choices[0].delta.content)
print('Marker:', marker)
print('Total content chars:', chars)

## TC-C4 — 對照組：**不帶** header（應只進 Solution ①，**不**進 EH）

In [ ]:
from openai import AzureOpenAI as _A
marker = f'{RUN_ID}-c4-controlgroup'
control = _A(
    azure_endpoint = APIM_ENDPOINT + APIM_API_PATH,
    api_key        = APIM_SUBSCRIPTION,
    api_version    = '2024-10-21',
    # 故意不帶 X-Logging-Channel
)
resp = control.chat.completions.create(
    model = DEPLOYMENT_NAME,
    messages = [{'role':'user','content': f'[{marker}] hello'}],
    max_tokens = 50,
)
print('Marker:', marker)
print('預期：EH/Blob 應**找不到**這個 marker；只能在 Solution ① 的 AppInsights 找到')

## 驗證 — 等 5 分鐘讓 Capture 觸發後查 Blob

Capture 預設 5 分鐘 / 300MB 觸發一次寫檔。下方 helper 直接列出 capture container 內 Avro 檔案。

In [ ]:
RG = 'newfoundry01'
STORAGE = subprocess.run(
    ['az','storage','account','list','-g',RG,'-o','json'],
    capture_output=True, text=True
).stdout
STORAGE = [s['name'] for s in json.loads(STORAGE) if s['name'].startswith('staigwc')][0]
print('Storage account:', STORAGE)

print('\nWaiting 5 minutes for first Capture flush...')
time.sleep(300)

out = subprocess.run([
    'az','storage','blob','list',
    '--account-name', STORAGE,
    '--container-name','capture',
    '--auth-mode','login',
    '--num-results','50',
    '-o','json'
], capture_output=True, text=True)
blobs = json.loads(out.stdout) if out.returncode == 0 else []
print(f'\nFound {len(blobs)} Avro file(s):')
for b in blobs[-10:]:
    print(f"  {b['name']}  ({b['properties']['contentLength']} bytes)")

## 下載最新 Avro 並解析 → 確認 marker 都在裡面（含未截斷的大 body）

In [ ]:
# pip install fastavro --quiet
import io, fastavro
if not blobs:
    print('沒有 blob，再等久一點或檢查 EH 是否有訊息')
else:
    latest = sorted(blobs, key=lambda b: b['properties']['lastModified'])[-1]['name']
    print('Reading:', latest)
    raw = subprocess.run([
        'az','storage','blob','download',
        '--account-name', STORAGE,
        '--container-name','capture',
        '--name', latest,
        '--auth-mode','login',
        '--no-progress','-f','-'
    ], capture_output=True).stdout
    reader = fastavro.reader(io.BytesIO(raw))
    events = []
    for rec in reader:
        try:
            body = json.loads(rec['Body'].decode('utf-8'))
            events.append(body)
        except Exception:
            pass
    print(f'Parsed {len(events)} events.')
    found = [e for e in events if e.get('correlationId','').startswith(RUN_ID)]
    print(f'Matching this RUN_ID ({RUN_ID}): {len(found)}')
    for e in found[:20]:
        print('  -', e.get('kind'), e.get('correlationId'),
              'chunk', e.get('chunkIndex'), '/', e.get('chunkTotal'),
              'len=', len(e.get('payload','')) if 'payload' in e else e.get('responseLength'))

## ✅ 驗收標準

| 項目 | 預期 |
|---|---|
| TC-C1 (短) | EH 內找得到 1 summary + 1 request-body chunk + 1 response-body chunk |
| TC-C2 (大) | response-body 拆 ≥ 2 chunk，總 length 大於 256KB（Solution ① 在此會被截）|
| TC-C3 (stream) | 仍能拿到完整 body（驗證 R4 在 Capture 路徑下表現）|
| TC-C4 (對照) | EH/Blob 內**沒有** `c4-controlgroup` 字樣 → 證明 header 隔離有效 |
| Solution ① | 全部 4 個 marker 都應在 AppInsights AppDependencies 找得到 → 證明雙軌獨立 |

## 後續查詢進階

如要長期 ad-hoc 查詢，建議掛 ADX external table 指向 capture container（範例：`kql/queries-eventhub.kql` C1-C5）。